# Flashify your own model and publish to HuggingFace

This notebook converts any RMSNorm-based HuggingFace checkpoint to a `-FlashNorm` variant (per-channel norm weights folded into the next linear layer, norm tensors removed from the state dict) and publishes it under your own HF account.

Works for Llama, Mistral, Gemma, Qwen, SmolLM, and any other transformer using RMSNorm followed by a linear layer.

Mathematically exact by Proposition 1 of the [FlashNorm paper](https://arxiv.org/abs/2407.09577).

## 1. Install and authenticate

Install the `transformer_tricks` package plus `huggingface_hub`, then log in with your HF write token so you can push to your own namespace.

In [ ]:
!pip install --quiet transformer-tricks huggingface-hub

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted (needs write scope for the target repo)

## 2. Configure source and destination

Set the source model on HuggingFace, the destination repo under your own account, and a local working directory.

In [ ]:
SRC = 'HuggingFaceTB/SmolLM2-135M'              # any RMSNorm-based HF model
OUT = 'YOUR_USERNAME/SmolLM2-135M-FlashNorm'    # destination (under your account)
LOCAL = './SmolLM2-135M_flashNorm'              # local workdir

## 3. Flashify

Fold the per-channel normalization weights `g` into the following linear layer (`W_star = W @ diag(g)`) and remove the now-constant norm tensors from the state dict. `strict=True` is the weightless format.

In [ ]:
import transformer_tricks as tt
tt.flashify_repo(SRC, dir=LOCAL, strict=True)

## 4. Upload to HuggingFace

Push the flashified checkpoint to your HF account. The repo is created automatically if it does not exist.

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(OUT, exist_ok=True)
api.upload_folder(repo_id=OUT, folder_path=LOCAL)
print(f'Published https://huggingface.co/{OUT}')

## 5. Verify

Load the checkpoint back from HF via `AutoModelForCausalLM`. You will see a warning about missing norm weights; Transformers defaults them to ones, which is the correct value for a FlashNorm checkpoint.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(OUT)
model = AutoModelForCausalLM.from_pretrained(OUT)

ids = tok('Once upon a time there was', return_tensors='pt').input_ids
out = model.generate(ids, max_new_tokens=30, do_sample=False)
print(tok.decode(out[0], skip_special_tokens=True))

## Framework support notes

- **HuggingFace Transformers**: works. Loads with a warning about missing norm weights; defaulted to ones, which is correct.
- **vLLM**: not yet. Tracking issue: TBD.
- **llama.cpp / GGUF**: not yet. Tracking issue: TBD.

For details on the transformation, see the [paper](https://github.com/OpenMachine-ai/transformer-tricks/blob/main/tex/flashNorm.tex) Section 3.1 and Proposition 1.